In [1]:
import fastbook
fastbook.setup_book()

from fastbook import *
from fastai.vision.widgets import *


In [ ]:
ims = search_images_ddg('grizzly bear')
len(ims)

In [ ]:
dest = 'images/grizzly.jpg'
download_url(ims[0],dest,show_progress=False)


In [ ]:
im = Image.open(dest)
im.to_thumb(128,128)

In [ ]:
bear_types = 'grizzly','black','teddy'
path = Path('bears')


In [ ]:
if not path.exists():
    path.mkdir()
    for o in bear_types:
        dest = (path/o)
        dest.mkdir(exist_ok=True)
        results = search_images_ddg(f'{o} bear')
        download_images(dest,urls=results)

In [ ]:
fns = get_image_files(path)
fns

In [ ]:
failed = verify_images(fns)
failed


In [ ]:
failed.map(Path.unlink)

In [ ]:
bears = DataBlock(

    blocks = (ImageBlock, CategoryBlock),
    get_items = get_image_files,
    splitter = RandomSplitter(valid_pct = 0.2, seed  = 42),
    get_y = parent_label,
    item_tfms=Resize(128)
       
) 

In [ ]:
dls = bears.dataloaders(path)

In [ ]:
dls.valid.show_batch(max_n=4, nrows=1)

In [ ]:
bears = bears.new(item_tfms=Resize(128,ResizeMethod.Pad,pad_mode='zeros'))
dls = bears.dataloaders(path)
dls.valid.show_batch(max_n=4, nrows=1)

In [ ]:
bears = bears.new(item_tfms=RandomResizedCrop(128,min_scale=0.3))
dls = bears.dataloaders(path)
dls.train.show_batch(max_n=4,nrows = 1,unique = True)

In [ ]:
bears = bears.new(item_tfms=Resize(128),batch_tfms=aug_transforms())
dls= bears.dataloaders(path)
dls.train.show_batch(max_n=8, nrows = 2, unique = True)

In [ ]:
bears = bears.new(

    items_tfms = RandomResizedCrop(224, min_scale = 0.5),
    batch_tfms=aug_transforms())

dls = bears.dataloaders(path)

In [ ]:
learn = vision_learner(dls,resnet18, metrics = error_rate)
learn.fine_tune(4)
